In [1]:
#Latex display work arround for VSCode
from latex_disp_VSCode import slatex

In [1]:
# Function to create a list of [x,y] variables names
# Input: n 
# Oupout: List of strings [x1, ... xn, y1, ..., yn], similarly for ZWvar and XZWvar
def XYvar(n):
    X=[]
    Y=[]
    for i in range(n):
        X.append('x'+str(i))
        Y.append('y'+str(i))
    X.extend(Y)
    return X

def ZWvar(n):
    Z=[]
    W=[]
    for i in range(n):
        Z.append('z'+str(i))
        W.append('w'+str(i))
    Z.extend(W)
    return Z

def XYZWvar(n, m):
    X=[]
    X = XYvar(n)
    X.extend(ZWvar(m))
    return X
# IMPORTANT: In this notebook, the variables previously defined will be used as variables for the systems of equations for Example 3.6

In [2]:
#Code to find solutions of superfrieze-systems in two odd variables \sigma \theta for the Kronecker triangulation for A(n,n), that is, the universal cover of said (unoriented) triangulation should be the same as the one showed in Figure 4. This code finds solutions using Gröebner basis, recall that if an ideal contains a constant, for instance 1, then is asscociated variety is empty. The code checks whether this is the case for the ideal generated by the system of equations associated to the Kronecker triangulation. 

#Inputs: 
# n is the number of different marked points (2n are the indices for x's and y's)
# m is the period of the superfrieze, it satisfies that n = k m for some k in Z
# A is a vector of the a_i coefficients of the m odd variables that are of the form ai \sigma + bi \theta
# B is a vector of the b_i coefficients of the m odd variables that are of the form ai \sigma + bi \theta
# L is the vector of the l_i coefficients that appear in the expression 3+ l\sigma\theta)
# A, B and L all must be of size m.

#Optional arguments
# groebner, always set to False. 
# ord, choice of monomial order, always set to 'degrevlex'
# helpb, always set to False, change to True if you want to see the reduction steps of the Groebner basis algorithm. If you want to the the steps, write
# ''from sage.misc.verbose import set_verbose
#   set_verbose(1)''
# at the beginning of your cell

# IMPORTANT: The algorithm used is the original Buchberger, it was used to be able to print the reduction steps. 

# Main tool: Grobner basis sage package

#Output: 
# If groebner = True, get the equations associated to the Kronecker superfrieze system. 
# Optional, if helpb = True, get the non-trivial steps of the Groebner basis algorithm. 
# Optional, if rexp = True, get only the truth value of the check if the system is solvable (True if it has a solution, False otherwise)
# Otherwise, the function outputs whether the system has a solution, by checking if 1 is in a grobner basis of the associated system. 

from sage.rings.polynomial.toy_buchberger import *
from sage.misc.verbose import set_verbose
from io import StringIO
import sys
def Kronecker_sys(A, B, L, n, groebner = False, ord = 'degrevlex', helpb = False, rexp = False):
    if len(A) != len(B):
        print ('A and B must be the sale lenght')
    else:
        m = len(A)
        if n % m != 0:
            print('m must divide n')
        else: 
            P = PolynomialRing(QQ, 4*n, XYvar(2*n), order=ord)
            P.inject_variables(verbose = False)
            eqsX = []
            eqsY = []
            eqsXY = []
            eqs = []
            nn=2*n
            for i in range(n):
                #append(P('x'+str((i)%n)+'+x'+str((i+1)%n)-A[i%m]))
                eqsX.append(P('x'+str((2*i)%nn)+'+x'+str((2*i+1)%nn)+'+x'+str((2*i+2)%nn))-A[i%m])
                eqsY.append(P('y'+str((2*i)%nn)+'+y'+str((2*i+1)%nn)+'+y'+str((2*i+2)%nn))-B[i%m])
                eqsXY.append(P('x'+str((2*i+1)%nn)+'*(y'+str((2*i+2)%nn)+'-y'+str((2*i)%nn)+')+y'+str((2*i+1)%nn)+'*(x'+str((2*i)%nn)+'-x'+str((2*i+2)%nn)+')+x'+str((2*i)%nn)+'*y'+str((2*i+2)%nn)+'-y'+str((2*i)%nn)+'*x'+str((2*i+2)%nn))-L[i%m])
            eqs.extend(eqsX)
            eqs.extend(eqsY)
            eqs.extend(eqsXY)
            if groebner :
                if helpb : 
                    buf = StringIO()
                    set_verbose(1)
                    old_stdout = sys.stdout
                    sys.stdout = buf
                    B = (P.ideal(eqs)).groebner_basis(algorithm = 'toy:buchberger')
                    sys.stdout = old_stdout
                    set_verbose(0)
                    for line in buf.getvalue().splitlines():
                        if '=> 0' not in line and 'G:' not in line and line.strip():  # keep only non-trivial
                            print(line)
                    return [eqs, B]
                else : 
                    B = (P.ideal(eqs)).groebner_basis(algorithm = 'toy:buchberger')
                    return [eqs, B]
            else:
                B = (P.ideal(eqs)).groebner_basis(algorithm = 'toy:buchberger')
                if rexp :
                    return not (1 in B)
                else : 
                    if 1 in B:
                        print('The system is not solvable')
                    else: 
                        print('The system has a solution')


In [3]:
# Example of system with no solution (compare with Theorem 3.7, here n < 3)
A=[2,2]
B=[3,3]
L=[0, 1]
Kronecker_sys(A, B, L, 2)
#If you want to see the elements of the groebner basis uncomment the following
#result = Kronecker_sys(A, B, L, 4, groebner=True)
#for i in range(len(result[1])):
#    print(result[1][i])


The system is not solvable


In [4]:
#The following coede let's check if random systems for the Kronecker case have a solution. 
# k - is the number of test you want to run
# q - is the range of the number of marked points of your annulus, that is, this code produces tests for A(l,l) with 2 \leq l \leq q-1
# p - is the range where you allow to take the coefficients A, B and L.
from sage.misc.prandom import randrange
from sage.misc.randstate import current_randstate

progress = {'i': 0, 'last': None}   # defined in global scope
def test_m(k, q, p) :
    for i in range(k):
        progress['i'] = i
        m = randint(2,q)
        A = [choice(list(range(1, p))) for j in range(m)]
        B = [choice(list(range(1, p))) for j in range(m)]
        L = [choice(list(range(1, p))) for j in range(m)]
        progress['last'] = (m, A, B, L)
        if not Kronecker_sys(A, B, L, m, rexp=True) : 
            print('This system has no solution')
            print([m,A,B,L])
            print('_________') 

#To run a test, uncomment the following line, it will only print systems that have no solution, it will only print systems where the number of marked points is less or equal than 2. 
#test_m(1000, 10, 10)

### Example of a non-realizable infinite superfrieze
As stated in Example 3.6, we construct superfriezes that cannot be realized from a triangulation of an annulus. The family we present here is of the form $A(n,k*n)$ triangulated as follows. Index the marked points in the inner boundary as $(i,j)$ where $1 \leq i \leq k$  and $1 \leq j \leq n$, order them in anticlockise order so that for fixed $j$, $(i+1,j)$ comes directly after $(i,j)$, and order them such that $(1,j+1)$ comes after $(k, j)$ (modulo $n$ in $j$ if needed). We triangulate $A(n,k*n)$ such that for a marked point $j$ in the outer boundary there are $k+1$ arcs to $(i,j)$ for all $1 \leq i \leq k$ as well as to $(1,j+1)$. 

Then $(1,j)$ is a fan centre who sees $3$ triangles, while the rest of the marked points in the inner boundary are adjacent two only $2$ triangles.

For a periodict infinite superfrieze whose associated integer frieze from comes from such a triangulation, the system of equations association to its realization problem is given by the following code. 

In [5]:
#General code for triangulations satisfying that the marked points in the inner boundary component are adjacent to at most 3 triangles, and such that the marked points with exacly 3 adjancet are never adjacent. We call this trangulation a "wedge" triangulation, where the wedges refer to the "upward" triangles, meaning those with to vertices lying in the outer boundary. We refer to "downward" triangles to the those with two vertices in the inner boundary. 
P = PolynomialRing(QQ, 40, XYZWvar(15, 5))
P.inject_variables(verbose = False)
def wedge_sys(A,B,L,n, m=1, N = []):
    """ System associated to a wedge triangulation.  
    n - total number of marked points in the inner boundary. This also gives the number of "downward" triangles. 
    m - number of wedges, default to 1
    N - vector of marked points where wedges in range(n) are (in order), default N=[0], racall that they cannot be adjacent. 
    """
    if len(A) != len(B):
        print ('A and B must be the same lenght')
    else:
        k = len(L)
        if n % k != 0:
            print('len(L) must divide n')
        else: 
            P = PolynomialRing(QQ, 2*n + 2*m, XYZWvar(n,m))
            P.inject_variables(verbose = False)
            N = [0]+N
            s = len(N)
            Np = [(N[i+1]-N[i]) for i in range(s-1)] + [n-N[s-1]]
            eqsX = []
            eqsY = []
            eqsXY = []
            eqsZ = []
            eqsZW = []
            eqs = []
            #print(N, Np)
            for i in range(s):
                r = N[i]
                for j in range(Np[i]-1):
                    eqsX.append(P('x'+str((j+r)%n)+'+x'+str((j+r+1)%n))-A[(j+r)%k])
                    eqsY.append(P('y'+str((j+r)%n)+'+y'+str((j+r+1)%n))-B[(j+r)%k])
                    eqsXY.append(P('x'+str((j+r)%n)+'*y'+str((j+r+1)%n)+'-x'+str((j+r+1)%n)+'*y'+str((j+r)%n))-L[(j+r)%k])
                eqsZ.append(P('x'+str((Np[i]-1+r)%n)+'+x'+str((Np[i]+r)%n)+'+z'+str(i%m))-A[(Np[i]-1+r)%k])
                eqsZ.append(P('y'+str((Np[i]-1+r)%n)+'+y'+str((Np[i]+r)%n)+'+w'+str(i%m))-B[(Np[i]-1+r)%k])
                eqsZW.append(P('x'+str((Np[i]-1+r)%n)+'*w'+str(i%s)+'-y'+str((Np[i]-1+r)%n)+'*z'+str(i%s)) + P('x'+str((Np[i]-1+r)%n)+'*y'+str((Np[i]+r)%n)+'-x'+str((Np[i]+r)%n)+'*y'+str((Np[i]-1+r)%n)) + P('-x'+str((Np[i]+r)%n)+'*w'+str(i%s)+'+y'+str((Np[i]+r)%n)+'*z'+str(i%s)) - L[(Np[i]-1+r)%k])
            eqs.extend(eqsX)
            eqs.extend(eqsY)
            eqs.extend(eqsXY)
            eqs.extend(eqsZ)
                #('x'+str((j+r)%n)+'*y'+str((j+r+1)%n)+'-x'+str((j+r+1)%n)+'*y'+str((j+r)%n))-L[(j+r)%k]
            eqs.extend(eqsZW)
            return eqs

In [6]:
#Example of such a system for a "wedge" triangulation of A(1,3). Compare this to the system in Example 3.6
A = [choice(list(range(1, 10))) for j in range(3)]
B = [choice(list(range(1, 10))) for j in range(3)]
L = [choice(list(range(1, 10))) for j in range(3)]
eqs = wedge_sys(A,B,L,3, 1)
A, B, L, eqs

([8, 1, 8],
 [5, 5, 2],
 [9, 5, 1],
 [x0 + x1 - 8,
  x1 + x2 - 1,
  y0 + y1 - 5,
  y1 + y2 - 5,
  -x1*y0 + x0*y1 - 9,
  -x2*y1 + x1*y2 - 5,
  x0 + x2 + z0 - 8,
  y0 + y2 + w0 - 2,
  x2*y0 - x0*y2 + y0*z0 - y2*z0 - x0*w0 + x2*w0 - 1])

In [7]:
#Groebner basis test:
GB = (P.ideal(eqs)).groebner_basis(algorithm = 'toy:buchberger')
not (1 in GB) # If False, the system has no solution (recall that if '1 in GB' is True then the associated variety is empty)

False

In [8]:
#To check whether the previous example is realizable as a A(k, 3*k)-superfrieze, you can run:
k = 2
Ap = A
Bp = B
Lp = L
for i in range(k-1):
    Ap = Ap + A
    Bp = Bp + B
    Lp = Lp + L
N = [3*j for j in range(1,k)]
eqsp = wedge_sys(A,B,L,3*k,k, N)
GBp = (P.ideal(eqsp)).groebner_basis(algorithm = 'toy:buchberger')
not (1 in GBp) # If False, the system has no solution (recall that if '1 in GB' is True then the associated variety is empty)

False